In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("cardiac_failure/responsivenes.csv")

# III.Hospitalization_discharge (Komal's dataset starts here)

# 1. Load and take a first look

In [30]:
df3 = pd.read_csv("cardiac_failure/hospitalization_discharge.csv")
print("Shape:", df3.shape)
df3.head()

Shape: (2008, 21)


,inpatient_number,destinationdischarge,admission_ward,admission_way,discharge_department,visit_times,respiratory_support,oxygen_inhalation,dischargeday,admission_date,...,death_within_28_days,re_admission_within_28_days,death_within_3_months,re_admission_within_3_months,death_within_6_months,re_admission_within_6_months,time_of_death__days_from_admission,readmission_time_days_from_admission,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months
0,857781,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,11,2017-01-24 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
1,743087,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,8,2017-05-05 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
2,866418,Home,Cardiology,NonEmergency,Cardiology,2,NaN,OxygenTherapy,5,2016-11-18 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,OxygenTherapy,11,2017-10-02 00:00:00,...,0,1,0,1,0,1,NaN,19.0,1.0,19.0
4,810128,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,5,2019-11-17 00:00:00,...,0,0,0,0,0,0,NaN,NaN,0.0,NaN


-The above code confirm with 2,008 hospitalization records across 21 variables — a mix of identifiers, categorical descriptors, and time-to-event outcomes. This is a classic survival/outcomes dataset structure (admission → discharge → downstream events like death or readmission), which shapes every decision that follows.

In [12]:
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 21 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   inpatient_number                                2008 non-null   int64  
 1   destinationdischarge                            2008 non-null   str    
 2   admission_ward                                  2008 non-null   str    
 3   admission_way                                   2008 non-null   str    
 4   discharge_department                            2008 non-null   str    
 5   visit_times                                     2008 non-null   int64  
 6   respiratory_support                             42 non-null     str    
 7   oxygen_inhalation                               2008 non-null   str    
 8   dischargeday                                    2008 non-null   int64  
 9   admission_date                                  2008

# 2. Check structure — duplicates and ID integrity

In [16]:
print("Fully duplicated rows:", df3.duplicated().sum())
print("Duplicate inpatient_number:", df3["inpatient_number"].duplicated().sum())

Fully duplicated rows: 0
Duplicate inpatient_number: 0


-Zero duplicate & inpatient_number values means each row is one hospitalization episode, not one patient (a patient could theoretically appear twice across different admissions — worth checking separately if patient-level analysis matters). Finding duplicates here would have signaled either a data export error or double-counted encounters, both of which would bias any incidence/rate calculation.

# 3. Handle missing values — deliberately, not blindly
*First inspect the pattern:

In [17]:
missing = df3.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df3) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

,missing_count,missing_pct
respiratory_support,1966,97.9
time_of_death__days_from_admission,1964,97.8
time_to_emergency_department_within_6_months,1111,55.3
readmission_time_days_from_admission,1107,55.1
return_to_emergency_department_within_6_months,1,0.0
discharge_department,0,0.0
admission_way,0,0.0
admission_ward,0,0.0
destinationdischarge,0,0.0
inpatient_number,0,0.0


In [18]:
df3["respiratory_support"] = df3["respiratory_support"].fillna("None")

-Respiratory_support being 98% empty doesn't mean 98% of records are broken — it means respiratory support (mechanical ventilation) is a rare intervention, and its absence is itself clinical information. Filling with "None" rather than dropping the column preserves that signal.

In [19]:
checks = [
    ("time_of_death__days_from_admission", "death_within_6_months"),
    ("readmission_time_days_from_admission", "re_admission_within_6_months"),
    ("time_to_emergency_department_within_6_months", "return_to_emergency_department_within_6_months"),
]
for time_col, flag_col in checks:
    mismatch = ((df3[flag_col] == 1) & (df3[time_col].isna())).sum()
    print(f"{time_col}: {mismatch} rows flagged '1' but missing a time value")

time_of_death__days_from_admission: 16 rows flagged '1' but missing a time value
readmission_time_days_from_admission: 1 rows flagged '1' but missing a time value
time_to_emergency_department_within_6_months: 7 rows flagged '1' but missing a time value


-The time-to-event columns (time_of_death, readmission_time, time_to_ED) being mostly empty reflects competing risks structure: most patients simply didn't have the event, so there's no time to record. This is normal in survival data, not a data quality failure.

In [21]:
df3[df3["return_to_emergency_department_within_6_months"].isna()]
df3["return_to_emergency_department_within_6_months"] = (
    df3["return_to_emergency_department_within_6_months"].fillna(0)
)

-The mismatch checks (16 death-flagged rows with no death time, 7 ED-flagged rows with no ED time) are the real find — these are genuine data entry gaps or logic errors that need manual review before you trust the death/readmission timing analysis. This is different in kind from the structural missingness above.

# 4. Fix data types

In [22]:
df3["admission_date"] = pd.to_datetime(df3["admission_date"], errors="coerce")
print("Unparsed dates:", df3["admission_date"].isna().sum())

binary_cols = [
    "death_within_28_days", "re_admission_within_28_days",
    "death_within_3_months", "re_admission_within_3_months",
    "death_within_6_months", "re_admission_within_6_months",
    "return_to_emergency_department_within_6_months",
]
df3[binary_cols] = df3[binary_cols].astype("Int64")

categorical_cols = [
    "destinationdischarge", "admission_ward", "admission_way",
    "discharge_department", "respiratory_support", "oxygen_inhalation",
    "outcome_during_hospitalization",
]
df3[categorical_cols] = df3[categorical_cols].astype("category")
df3.dtypes

Unparsed dates: 0


inpatient_number                                           int64
destinationdischarge                                    category
admission_ward                                          category
admission_way                                           category
discharge_department                                    category
visit_times                                                int64
respiratory_support                                     category
oxygen_inhalation                                       category
dischargeday                                               int64
admission_date                                    datetime64[us]
outcome_during_hospitalization                          category
death_within_28_days                                       Int64
re_admission_within_28_days                                Int64
death_within_3_months                                      Int64
re_admission_within_3_months                               Int64
death_within_6_months    

-Casting binary flags to Int64 (nullable) rather than plain int preserves the ability to distinguish "confirmed 0" from "unknown" if that ever comes up. Casting categoricals lets pandas store and group them efficiently and exposes the full category list at a glance.

# 5. Check categorical consistency

In [23]:
for col in categorical_cols:
    print(col, "->", list(df3[col].cat.categories))

destinationdischarge -> ['Died', 'HealthcareFacility', 'Home', 'Unknown']
admission_ward -> ['Cardiology', 'GeneralWard', 'ICU', 'Others']
admission_way -> ['Emergency', 'NonEmergency']
discharge_department -> ['Cardiology', 'GeneralWard', 'ICU', 'Others']
respiratory_support -> ['IMV', 'NIMV', 'None']
oxygen_inhalation -> ['AmbientAir', 'OxygenTherapy']
outcome_during_hospitalization -> ['Alive', 'Dead', 'DischargeAgainstOrder']


-Printing each column's unique categories confirms there's no accidental fragmentation — e.g., "Home" vs "home" vs "Home " being treated as three different discharge destinations. In this dataset it came back clean (4 destinations, 4 wards, 2 admission ways, 3 outcomes), which tells you the source system enforces controlled vocabularies — a good sign for downstream modeling since you won't need fuzzy-matching or recoding.

# 6. Flag outliers for review 

In [24]:
numeric_cols = ["visit_times", "dischargeday"]
for col in numeric_cols:
    q1, q3 = df3[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df3[(df3[col] < lower) | (df3[col] > upper)]
    print(f"{col}: {len(outliers)} potential outliers (outside [{lower:.1f}, {upper:.1f}])")

visit_times: 148 potential outliers (outside [1.0, 1.0])
dischargeday: 163 potential outliers (outside [0.0, 16.0])


-Dischargeday outliers (163 rows outside ~16 days) aren't necessarily errors — a max of 123 days could reflect a genuinely complex, prolonged case (ICU complications, comorbidities). The point of flagging rather than deleting is that in clinical data, extreme values often carry the most important signal (sickest patients), so removing them would bias the dataset toward "easy" cases. visit_times outliers (148 rows) flag patients admitted more than once in this dataset — potentially useful as a frailty/chronicity indicator rather than noise.

# 7. Cross-field logic checks

In [25]:
bad_death = df3[(df3["death_within_28_days"] == 1) & (df3["death_within_6_months"] == 0)]
bad_readm = df3[(df3["re_admission_within_28_days"] == 1) & (df3["re_admission_within_6_months"] == 0)]
print("28-day death not reflected at 6 months:", len(bad_death))
print("28-day readmission not reflected at 6 months:", len(bad_readm))

28-day death not reflected at 6 months: 0
28-day readmission not reflected at 6 months: 3


-The 3 rows where a 28-day readmission isn't reflected as a 6-month readmission are a logical impossibility (6-month window should be a superset of 28-day), not clinical variation. This is the kind of finding pure missing-value or outlier analysis won't catch — it only shows up when you check that the relationships between columns are internally consistent. This is often where real transcription or ETL errors hide.

# 8. Export

In [26]:
df3.to_csv("cardiachospitalization_discharge_cleaned.csv", index=False)

-Locking in a cleaned, typed, structurally-verified file separates the "data cleaning" phase from the "analysis" phase — so any modeling or stats you run later starts from a known-good, reproducible baseline rather than re-deriving these decisions each time.

# Komal code ends here

In [5]:
df.head()


,inpatient_number,eye_opening,verbal_response,movement,consciousness,gcs
0,857781,4,5,6,Clear,15
1,743087,4,5,6,Clear,15
2,866418,4,5,6,Clear,15
3,775928,4,5,6,Clear,15
4,810128,4,5,6,Clear,15


In [15]:
df.shape

(2008, 6)

In [16]:
df.columns.tolist()

['inpatient_number',
 'eye_opening',
 'verbal_response',
 'movement',
 'consciousness',
 'gcs']

In [17]:
df.isnull().sum()

inpatient_number    0
eye_opening         0
verbal_response     0
movement            0
consciousness       0
gcs                 0
dtype: int64

In [18]:
df['inpatient_number'].duplicated().sum()

np.int64(0)

In [19]:
df.dtypes


inpatient_number    int64
eye_opening         int64
verbal_response     int64
movement            int64
consciousness         str
gcs                 int64
dtype: object

In [20]:
print("Eye Opening:")
print(sorted(df['eye_opening'].unique()))

print("\nVerbal Response:")
print(sorted(df['verbal_response'].unique()))

print("\nMovement:")
print(sorted(df['movement'].unique()))

print("\nGCS:")
print(sorted(df['gcs'].unique()))

print("\nConsciousness:")
print(df['consciousness'].unique())

Eye Opening:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Verbal Response:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Movement:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

GCS:
[np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

Consciousness:
<ArrowStringArray>
['Clear', 'ResponsiveToPain', 'ResponsiveToSound', 'Nonresponsive']
Length: 4, dtype: str


In [21]:
df['calculated_gcs'] = (
    df['eye_opening']
    + df['verbal_response']
    + df['movement']
)

df[df['calculated_gcs'] != df['gcs']]

,inpatient_number,eye_opening,verbal_response,movement,consciousness,gcs,calculated_gcs


In [22]:
len(df[df['calculated_gcs'] != df['gcs']])


0

In [23]:
df.drop(columns=['calculated_gcs'], inplace=True)

In [24]:
df.shape

(2008, 6)

In [25]:
df['gcs'].value_counts().sort_index()


gcs
3       13
4        1
6        1
7        4
10       7
11      19
12       3
13       2
14       7
15    1951
Name: count, dtype: int64

In [26]:
df['consciousness'].value_counts()


consciousness
Clear                1974
ResponsiveToSound      19
Nonresponsive          11
ResponsiveToPain        4
Name: count, dtype: int64

In [27]:
df['eye_opening'].value_counts().sort_index()

eye_opening
1      14
2       3
3      25
4    1966
Name: count, dtype: int64